In [4]:
import pandas as pd
import httpx

In [ ]:
import asyncio

HEADERS = {
    'accept': '*/*',
    'accept-language': 'en-IN,en-GB;q=0.9,en-US;q=0.8,en;q=0.7',
    'origin': 'https://www.geeksforgeeks.org',
    'priority': 'u=1, i',
    'referer': 'https://www.geeksforgeeks.org/',
    'sec-ch-ua': '"Chromium";v="148", "Google Chrome";v="148", "Not/A)Brand";v="99"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Linux"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-site',
    'user-agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36',
}

BATCH_SIZE = 50
sem = asyncio.Semaphore(10)

async def fetch_page(client, page):
    async with sem:
        resp = await client.get(
            'https://practiceapi.geeksforgeeks.org/api/vr/problems/',
            params={'pageMode': 'explore', 'page': page},
            headers=HEADERS,
        )
        return resp.json().get('results', [])

async def fetch_all():
    async with httpx.AsyncClient() as client:
        questions = []
        batch_start = 0
        while True:
            batch = await asyncio.gather(
                *[fetch_page(client, p) for p in range(batch_start, batch_start + BATCH_SIZE)]
            )
            done = False
            for i, results in enumerate(batch):
                if not results and i + 1 < len(batch) and not batch[i + 1]:
                    done = True
                    break
                questions.extend(results)
            print(f"Batch {batch_start // BATCH_SIZE}: {len(questions)} questions so far")
            if done:
                break
            batch_start += BATCH_SIZE
    return questions

questions = await fetch_all()


In [10]:
response.json()

{'previous': 2,
 'next': 4,
 'count': 20,
 'total': 3740,
 'solved': 0,
 'unsolved': 0,
 'results': [{'id': 700099,
   'problem_name': 'Detect Loop in linked list',
   'problem_type': 1,
   'problem_level': 1,
   'slug': 'detect-loop-in-linked-list',
   'accuracy': '43.49%',
   'all_submissions': 526437,
   'marks': 4,
   'difficulty': 'Medium',
   'tags': {'company_tags': ['Paytm',
     'VMWare',
     'Accolite',
     'Amazon',
     'OYO Rooms',
     'Samsung',
     'Snapdeal',
     'D-E-Shaw',
     'Hike',
     'MakeMyTrip',
     'Walmart',
     'MAQ Software',
     'Adobe',
     'SAP Labs',
     'Qualcomm',
     'Veritas',
     'Mahindra Comviva',
     'Lybrate'],
    'topic_tags': ['Linked List',
     'two-pointer-algorithm',
     'Data Structures',
     'Algorithms']},
   'content_type': 1,
   'problem_url': 'https://www.geeksforgeeks.org/problems/detect-loop-in-linked-list/1',
   'topic_order': None,
   'visibility_type': 1,
   'batch_slug': None,
   'track_slug': None,
   'solve

In [ ]:
import itertools
import json
from pathlib import Path

import tqdm.asyncio

SAVE_DIR = Path("data/gfg")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

scrape_sem = asyncio.Semaphore(30)


async def fetch_and_save(client: httpx.AsyncClient, question: dict):
    slug = question["slug"]
    out_path = SAVE_DIR / f"{slug}.json"
    if out_path.exists():
        return

    async with scrape_sem:
        try:
            resp = await client.get(question["problem_url"], follow_redirects=True)
            resp.raise_for_status()
        except Exception as e:
            print(f"Failed {slug}: {e}")
            return

    out_path.write_text(json.dumps({**question, "html": resp.text}, ensure_ascii=False))


async def scrape_all():
    clients = [
        httpx.AsyncClient(proxy=p, timeout=30) for p in list_proxy
    ]
    client_cycle = itertools.cycle(clients)
    try:
        tasks = [fetch_and_save(next(client_cycle), q) for q in questions]
        await tqdm.asyncio.tqdm.gather(*tasks)
    finally:
        await asyncio.gather(*[c.aclose() for c in clients])


await scrape_all()
